In [1]:
import contextlib
from pathlib import Path
from uuid import uuid4

import plotly.figure_factory as ff
import plotly.graph_objects as go
import polars as pl
import torch
import torch.nn.functional as F
from IPython.display import display
from lightning import Fabric
from tqdm.autonotebook import tqdm

from src.chart_transport.constraint import (
    LagrangianConstraintConfig,
    LossConstraintConfig,
)
from src.common.training import TrainingConfig
from src.config.base import BaseConfig
from src.data.gaussian_mixture.data import MultimodalGaussianDataConfig
from src.experiments.multimodal_gaussian.canonical import (
    get_canonical_chart_transport_configs,
)
from src.experiments.multimodal_gaussian.plotting.constraints import (
    plot_latents,
    plot_sample_pairs,
)

torch.set_float32_matmul_precision("medium")


In [2]:
class RunConfig(BaseConfig):
    seed: int
    integrated_n_steps: int


run_config = RunConfig(
    seed=0,
    integrated_n_steps=4_000,
)

num_modes = 4
mode_std = 0.1
offset = 1.0
ambient_dimension = 8

data_config = MultimodalGaussianDataConfig.initialize(
    num_modes=num_modes,
    mode_std=mode_std,
    offset=offset,
    ambient_dimension=ambient_dimension,
)

chart_transport_config = get_canonical_chart_transport_configs(
    data_config=data_config,
)

training_config = TrainingConfig.initialize(
    train_batch_size=2048,
    eval_batch_size=1024,
    eval_every_n_training_steps=3000,
    folder=(
        Path("/home/nlyu/Code/diffusive-latent-generation/artifacts/multimodal_gaussian")
        / uuid4().hex
    ),
)

chart_transport_config.visualize()


In [3]:
fabric = Fabric(
    accelerator="cuda", devices=1,
    precision="bf16-mixed",
)
fabric.launch()
fabric.seed_everything(run_config.seed)

device = fabric.device
device_type = device.type

runtime_data_config = data_config.replace(
    path="isometry",
    replacement=data_config.isometry.to(device=device, dtype=torch.float32),
).replace(
    path="projection",
    replacement=data_config.projection.to(device=device, dtype=torch.float32),
)

architecture_config = chart_transport_config.architecture_config
chart_transport_model = architecture_config.get_model()
optimizer = architecture_config.get_optimizer(
    model=chart_transport_model,
)
chart_transport_model, optimizer = fabric.setup(chart_transport_model, optimizer)

encoder = chart_transport_model.encoder
decoder = chart_transport_model.decoder
critic = chart_transport_model.critic

prior_config = chart_transport_config.prior_config
constraint_config = chart_transport_config.loss_config.constraint_config
constraint_method = constraint_config.constraint_method
pretrain_config = chart_transport_config.loss_config.chart_pretrain_config
transport_config = chart_transport_config.loss_config.transport_config
transport_t_min, transport_t_max = transport_config.t_range

data_dual = torch.zeros((), device=device)
prior_dual = torch.zeros((), device=device)

fabric.print(
    f"device={device}, precision=bf16-mixed, folder={training_config.folder}"
)

Using bfloat16 Automatic Mixed Precision (AMP)
Seed set to 0


device=cuda:0, precision=bf16-mixed, folder=/home/nlyu/Code/diffusive-latent-generation/artifacts/multimodal_gaussian/f6390868a3f94bba929d2e9be37248c8


In [ ]:
class PretrainMonitorConfig(BaseConfig):
    log_every_n_steps: int
    plot_n_sample_pairs: int
    plot_n_data_latents_per_mode: int


class CriticMonitorConfig(BaseConfig):
    """
    Config for visualizing the score implied by the critic. By one plot, we mean "html + png"
    1. Saves one plot for each sample_t_values, with two arrows attached to each point
        corresponding to the "data score" and the "prior - data score" at the noise level
    2. Saves one plot "transport "
        corresponding to the implied drift field for **clean data latent**,
        averaged across the noise spectrum. Background corresponds to contour lines
    3. Saves one plot "loss_spectrum" with x-axis being losses at
        unif[0, 1, interval=0.05] corresponding to the **score matching** loss at each point.
        Read the theory very carefully, I think this corresponds to noise-pred mse * (1-t)**2 / t ** 2
    Keep these specs after modification
    """

    log_every_n_steps: int #  To delete
    sample_t_values: list[float]
    num_contour_lines: int
    plot_n_data_latents_per_mode: int
    plot_n_vectors_per_mode: int


class IntegratedMonitorConfig(BaseConfig):
    """
    Integrated training needs to execute pretrain-monitor, critic-monitor, together with:
    save a scatterplot of the generated samples.
    Also save plotly plots (png + svg) of:
    1. Dual variables (data + prior) on the right axis, together with loss (data / prior) annotated on the left axis
    2. Transport field norm at step (right axis), avg log likelihood of batch samples (left axis)
    3. Plot of the score critic loss across steps
    """
    log_every_n_steps: int # To delete
    plot_n_generated_samples: int # 3000
    keep_loss_last_n: int # 2000


class MonitorConfig(BaseConfig):
    pretrain_monitor_config: PretrainMonitorConfig
    critic_monitor_config: CriticMonitorConfig
    integrated_monitor_config: IntegratedMonitorConfig

    log_every_n_steps_manifold_pretrain: int # 500
    log_every_n_steps_critic_pretrain: int # 500
    log_every_n_steps_integrated_transport: int # 200


monitor_config = MonitorConfig(
    pretrain_monitor_config=PretrainMonitorConfig(
        log_every_n_steps=500,
        plot_n_sample_pairs=1_000,
        plot_n_data_latents_per_mode=1_000,
    ),
    critic_monitor_config=CriticMonitorConfig(
        log_every_n_steps=500,
        sample_t_values=[0.03, 0.2, 0.5],
        num_contour_lines=10,
        plot_n_data_latents_per_mode=1_000,
        plot_n_vectors_per_mode=50,
    ),
    integrated_monitor_config=IntegratedMonitorConfig(
        log_every_n_steps=500,
    ),
)
monitor_config.visualize()


In [5]:
MODE_COLORS = (
    "#1f77b4",
    "#ff7f0e",
    "#2ca02c",
    "#d62728",
    "#9467bd",
    "#8c564b",
)


def sample_monitor_batch(
    *,
    batch_size_per_mode: int,
) -> tuple[torch.Tensor, torch.Tensor]:
    batches = []
    labels = []
    for mode_id in range(runtime_data_config.num_modes):
        batches.append(
            runtime_data_config.sample_class(
                mode_id=mode_id,
                batch_size=batch_size_per_mode,
            )
        )
        labels.append(
            torch.full(
                (batch_size_per_mode,),
                fill_value=mode_id,
                device=device,
                dtype=torch.long,
            )
        )
    return torch.cat(batches, dim=0), torch.cat(labels, dim=0)


def sample_monitor_pairs_batch(
    *,
    total_batch_size: int,
) -> tuple[torch.Tensor, torch.Tensor]:
    batch_size_per_mode = max(
        1,
        (total_batch_size + runtime_data_config.num_modes - 1)
        // runtime_data_config.num_modes,
    )
    samples, labels = sample_monitor_batch(
        batch_size_per_mode=batch_size_per_mode,
    )
    return samples[:total_batch_size], labels[:total_batch_size]


def write_plot_artifacts(
    *,
    figure,
    step: int,
    plot_type: str,
    stage: str = "pretrain",
) -> None:
    output_folder = training_config.folder / stage / str(step)
    output_folder.mkdir(parents=True, exist_ok=True)
    figure.write_html(output_folder / f"{plot_type}.html")
    figure.write_image(output_folder / f"{plot_type}.png")


def latent_square_limits(
    points: torch.Tensor,
    *,
    padding: float = 0.18,
) -> tuple[float, float, float, float]:
    mins = points.min(dim=0).values
    maxs = points.max(dim=0).values
    center = 0.5 * (mins + maxs)
    radius = 0.5 * float((maxs - mins).max().item())
    radius = max(radius * (1.0 + padding), 1.0)
    return (
        float(center[0] - radius),
        float(center[0] + radius),
        float(center[1] - radius),
        float(center[1] + radius),
    )


def build_latent_grid(
    *,
    points: torch.Tensor,
    resolution: int,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    x_min, x_max, y_min, y_max = latent_square_limits(points)
    xs = torch.linspace(
        x_min,
        x_max,
        resolution,
        device=points.device,
        dtype=points.dtype,
    )
    ys = torch.linspace(
        y_min,
        y_max,
        resolution,
        device=points.device,
        dtype=points.dtype,
    )
    grid_y, grid_x = torch.meshgrid(ys, xs, indexing="ij")
    grid_points = torch.stack([grid_x.reshape(-1), grid_y.reshape(-1)], dim=-1)
    return grid_points, xs, ys


def vector_display_length(
    points: torch.Tensor,
    *,
    fraction: float = 0.02,
) -> float:
    x_min, x_max, y_min, y_max = latent_square_limits(points, padding=0.0)
    return fraction * max(x_max - x_min, y_max - y_min)


def normalize_vectors(
    *,
    vectors: torch.Tensor,
    display_length: float,
) -> torch.Tensor:
    magnitudes = vectors.norm(dim=-1, keepdim=True)
    return display_length * vectors / magnitudes.clamp_min(1e-6)


def add_mode_scatter_traces(
    *,
    figure: go.Figure,
    points: torch.Tensor,
    labels: torch.Tensor,
    size: float,
    opacity: float,
    showlegend: bool,
    line_width: float = 0.0,
) -> None:
    points_cpu = points.detach().cpu().float()
    labels_cpu = labels.detach().cpu().long()
    for mode_id in range(runtime_data_config.num_modes):
        mask = labels_cpu == mode_id
        if not mask.any():
            continue
        figure.add_trace(
            go.Scatter(
                x=points_cpu[mask, 0].tolist(),
                y=points_cpu[mask, 1].tolist(),
                mode="markers",
                name=f"mode {mode_id}",
                showlegend=showlegend,
                marker={
                    "size": size,
                    "opacity": opacity,
                    "color": MODE_COLORS[mode_id % len(MODE_COLORS)],
                    "line": {
                        "width": line_width,
                        "color": "rgba(0, 0, 0, 0.30)",
                    },
                },
            )
        )


def add_mode_quiver_traces(
    *,
    figure: go.Figure,
    points: torch.Tensor,
    vectors: torch.Tensor,
    labels: torch.Tensor,
) -> None:
    points_cpu = points.detach().cpu().float()
    vectors_cpu = vectors.detach().cpu().float()
    labels_cpu = labels.detach().cpu().long()
    for mode_id in range(runtime_data_config.num_modes):
        mask = labels_cpu == mode_id
        if not mask.any():
            continue
        quiver = ff.create_quiver(
            x=points_cpu[mask, 0].tolist(),
            y=points_cpu[mask, 1].tolist(),
            u=vectors_cpu[mask, 0].tolist(),
            v=vectors_cpu[mask, 1].tolist(),
            scale=1.0,
            arrow_scale=0.25,
            line_color=MODE_COLORS[mode_id % len(MODE_COLORS)],
            name=f"mode {mode_id}",
        )
        for trace in quiver.data:
            trace.showlegend = False
            trace.hoverinfo = "skip"
            trace.line.width = 1.2
            figure.add_trace(trace)


def sample_transport_times(
    *,
    batch_shape: tuple[int, ...],
) -> torch.Tensor:
    return transport_t_min + (transport_t_max - transport_t_min) * torch.rand(
        *batch_shape,
        device=device,
        dtype=torch.float32,
    )


def sample_stratified_transport_times(
    *,
    batch_size: int,
    num_time_samples: int,
) -> torch.Tensor:
    bin_edges = torch.linspace(
        transport_t_min,
        transport_t_max,
        num_time_samples + 1,
        device=device,
        dtype=torch.float32,
    )
    bin_starts = bin_edges[:-1]
    bin_widths = bin_edges[1:] - bin_edges[:-1]
    return bin_starts.unsqueeze(0) + bin_widths.unsqueeze(0) * torch.rand(
        batch_size,
        num_time_samples,
        device=device,
        dtype=torch.float32,
    )


def critic_score_from_noise_prediction(
    *,
    predicted_noise: torch.Tensor,
    t: torch.Tensor,
) -> torch.Tensor:
    return -predicted_noise / t.unsqueeze(-1)


def critic_spectrum_t_values() -> torch.Tensor:
    return torch.linspace(
        transport_t_min,
        transport_t_max,
        19,
        device=device,
        dtype=torch.float32,
    )


def sample_critic_snapshot(
    *,
    clean_latents: torch.Tensor,
    t_value: float,
) -> tuple[torch.Tensor, torch.Tensor]:
    t = torch.full(
        (clean_latents.shape[0],),
        float(t_value),
        device=clean_latents.device,
        dtype=torch.float32,
    )
    noise = torch.randn_like(clean_latents)
    noised_latents = (
        (1.0 - t).unsqueeze(-1) * clean_latents + t.unsqueeze(-1) * noise
    )
    predicted_noise = critic(noised_latents, t).float()
    data_score = critic_score_from_noise_prediction(
        predicted_noise=predicted_noise,
        t=t,
    )
    return noised_latents.float(), data_score.float()


def estimate_clean_transport_field(
    *,
    clean_latents: torch.Tensor,
    t_values: torch.Tensor,
) -> torch.Tensor:
    transport_field = torch.zeros_like(clean_latents, dtype=torch.float32)
    for t_value in t_values.detach().cpu().tolist():
        t = torch.full(
            (clean_latents.shape[0],),
            float(t_value),
            device=clean_latents.device,
            dtype=torch.float32,
        )
        pullback_weight = transport_config.kl_weight_schedule.pullback_weight(
            t.float(),
        ).unsqueeze(-1)
        noise = torch.randn_like(clean_latents)

        def evaluate_with_noise(sampled_noise: torch.Tensor) -> torch.Tensor:
            noised_latents = (
                (1.0 - t).unsqueeze(-1) * clean_latents
                + t.unsqueeze(-1) * sampled_noise
            )
            predicted_noise = critic(noised_latents, t).float()
            prior_score = prior_config.analytic_score(
                noised_latents.float(),
                t.float(),
            ).float()
            return pullback_weight * (
                prior_score
                + predicted_noise / t.unsqueeze(-1)
            )

        transport_terms = evaluate_with_noise(noise)
        if transport_config.antipodal_estimate:
            transport_terms = 0.5 * (
                transport_terms + evaluate_with_noise(-noise)
            )
        transport_field = transport_field + transport_terms
    return transport_field / t_values.numel()


def estimate_critic_loss_spectrum(
    *,
    clean_latents: torch.Tensor,
    t_values: torch.Tensor,
) -> pl.DataFrame:
    rows = []
    for t_value in t_values.detach().cpu().tolist():
        t = torch.full(
            (clean_latents.shape[0],),
            float(t_value),
            device=clean_latents.device,
            dtype=torch.float32,
        )
        noise = torch.randn_like(clean_latents)

        def mse_with_noise(sampled_noise: torch.Tensor) -> torch.Tensor:
            noised_latents = (
                (1.0 - t).unsqueeze(-1) * clean_latents
                + t.unsqueeze(-1) * sampled_noise
            )
            predicted_noise = critic(noised_latents, t).float()
            return (predicted_noise - sampled_noise.float()).square().mean()

        noise_prediction_mse = mse_with_noise(noise)
        if transport_config.antipodal_estimate:
            noise_prediction_mse = 0.5 * (
                noise_prediction_mse + mse_with_noise(-noise)
            )

        rows.append(
            {
                "t": float(t_value),
                "noise_prediction_mse": float(noise_prediction_mse.item()),
            }
        )
    return pl.DataFrame(rows)


def plot_critic_score_snapshot(
    *,
    cloud_latents: torch.Tensor,
    cloud_labels: torch.Tensor,
    arrow_latents: torch.Tensor,
    arrow_labels: torch.Tensor,
    data_score: torch.Tensor,
    t_value: float,
) -> go.Figure:
    if cloud_latents.shape[-1] != 2:
        raise ValueError("Critic monitor plots require 2D latent coordinates")

    figure = go.Figure()
    add_mode_scatter_traces(
        figure=figure,
        points=cloud_latents,
        labels=cloud_labels,
        size=5,
        opacity=0.28,
        showlegend=True,
    )
    add_mode_scatter_traces(
        figure=figure,
        points=arrow_latents,
        labels=arrow_labels,
        size=5,
        opacity=0.28,
        showlegend=False,
    )
    add_mode_quiver_traces(
        figure=figure,
        points=arrow_latents,
        vectors=normalize_vectors(
            vectors=data_score,
            display_length=vector_display_length(cloud_latents),
        ),
        labels=arrow_labels,
    )
    x_min, x_max, y_min, y_max = latent_square_limits(cloud_latents)
    figure.update_layout(
        title=f"Critic score snapshot at t={t_value:.2f}",
        template="plotly_white",
        width=850,
        height=850,
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0.0},
        xaxis={"range": [x_min, x_max]},
        yaxis={"range": [y_min, y_max], "scaleanchor": "x", "scaleratio": 1.0},
    )
    return figure


def plot_transport_field(
    *,
    cloud_latents: torch.Tensor,
    cloud_labels: torch.Tensor,
    arrow_latents: torch.Tensor,
    arrow_labels: torch.Tensor,
    transport_field: torch.Tensor,
    grid_xs: torch.Tensor,
    grid_ys: torch.Tensor,
    grid_transport_projection: torch.Tensor,
) -> go.Figure:
    if cloud_latents.shape[-1] != 2:
        raise ValueError("Critic monitor plots require 2D latent coordinates")

    figure = go.Figure()
    figure.add_trace(
        go.Contour(
            x=grid_xs.detach().cpu().float().tolist(),
            y=grid_ys.detach().cpu().float().tolist(),
            z=grid_transport_projection.detach().cpu().float().numpy(),
            contours={"coloring": "lines", "showlabels": True},
            line={"width": 1.1, "color": "rgba(70, 70, 70, 0.55)"},
            ncontours=monitor_config.critic_monitor_config.num_contour_lines,
            showscale=False,
            name="transport norm contours",
            hoverinfo="skip",
        )
    )
    add_mode_scatter_traces(
        figure=figure,
        points=cloud_latents,
        labels=cloud_labels,
        size=5,
        opacity=0.28,
        showlegend=True,
    )
    add_mode_scatter_traces(
        figure=figure,
        points=arrow_latents,
        labels=arrow_labels,
        size=5,
        opacity=0.28,
        showlegend=False,
    )
    add_mode_quiver_traces(
        figure=figure,
        points=arrow_latents,
        vectors=normalize_vectors(
            vectors=transport_field,
            display_length=vector_display_length(cloud_latents),
        ),
        labels=arrow_labels,
    )
    x_min, x_max, y_min, y_max = latent_square_limits(cloud_latents)
    figure.update_layout(
        title="Noise-averaged clean-latent transport field",
        template="plotly_white",
        width=850,
        height=850,
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0.0},
        xaxis={"range": [x_min, x_max]},
        yaxis={"range": [y_min, y_max], "scaleanchor": "x", "scaleratio": 1.0},
    )
    return figure


def plot_critic_loss_spectrum(
    *,
    loss_spectrum: pl.DataFrame,
) -> go.Figure:
    figure = go.Figure()
    figure.add_trace(
        go.Scatter(
            x=loss_spectrum["t"].to_list(),
            y=loss_spectrum["noise_prediction_mse"].to_list(),
            mode="lines+markers",
            line={"color": "#1f77b4", "width": 2.0},
            marker={"size": 7},
            name="noise prediction mse",
        )
    )
    figure.update_layout(
        title="Critic loss spectrum",
        template="plotly_white",
        width=900,
        height=500,
        xaxis={},
        yaxis={},
    )
    return figure


def monitor_pretrain_step(
    *,
    step: int,
) -> None:
    encoder_was_training = encoder.training
    decoder_was_training = decoder.training
    encoder.eval()
    decoder.eval()
    with torch.no_grad():
        pair_samples, pair_labels = sample_monitor_pairs_batch(
            total_batch_size=monitor_config.pretrain_monitor_config.plot_n_sample_pairs,
        )
        pair_latents = encoder(pair_samples)
        pair_reconstructions = decoder(pair_latents)
        pair_projected_samples, _, _ = data_config.decompose_projection(
            pair_samples.float().cpu(),
        )
        pair_projected_reconstructions, _, _ = data_config.decompose_projection(
            pair_reconstructions.float().cpu(),
        )
        pair_manifold_deviation = (
            (pair_reconstructions - pair_samples)
            .reshape(pair_samples.shape[0], -1)
            .norm(dim=-1)
            .float()
            .cpu()
        )

        latent_samples, latent_labels = sample_monitor_batch(
            batch_size_per_mode=monitor_config.pretrain_monitor_config.plot_n_data_latents_per_mode,
        )
        latent_values = encoder(latent_samples).float().cpu()
        _, _, latent_off_plane = data_config.decompose_projection(
            latent_samples.float().cpu(),
        )
        latent_off_manifold_norm = latent_off_plane.norm(dim=-1)

    if encoder_was_training:
        encoder.train()
    if decoder_was_training:
        decoder.train()

    sample_pairs_figure = plot_sample_pairs(
        samples=pair_projected_samples,
        pairs=pair_projected_reconstructions,
        manifold_deviation=pair_manifold_deviation,
        labels=pair_labels.cpu(),
    )
    write_plot_artifacts(
        figure=sample_pairs_figure,
        step=step,
        plot_type="sample_pairs",
        stage="pretrain",
    )

    latents_figure = plot_latents(
        latents=latent_values,
        off_manifold_norm=latent_off_manifold_norm,
        labels=latent_labels.cpu(),
    )
    write_plot_artifacts(
        figure=latents_figure,
        step=step,
        plot_type="latents",
        stage="pretrain",
    )


def monitor_critic_step(
    *,
    step: int,
) -> None:
    encoder_was_training = encoder.training
    critic_was_training = critic.training
    encoder.eval()
    critic.eval()

    with torch.no_grad():
        with contextlib.ExitStack() as stack:
            if device_type == "cuda":
                stack.enter_context(torch.cuda.device(device))
            stack.enter_context(torch.device(str(device)))
            stack.enter_context(torch.autocast(device_type=device_type, dtype=torch.bfloat16))

            dense_samples, dense_labels = sample_monitor_batch(
                batch_size_per_mode=monitor_config.critic_monitor_config.plot_n_data_latents_per_mode,
            )
            vector_samples, vector_labels = sample_monitor_batch(
                batch_size_per_mode=monitor_config.critic_monitor_config.plot_n_vectors_per_mode,
            )
            dense_clean_latents = encoder(dense_samples).float()
            vector_clean_latents = encoder(vector_samples).float()

            for t_value in monitor_config.critic_monitor_config.sample_t_values:
                cloud_latents, _ = sample_critic_snapshot(
                    clean_latents=dense_clean_latents,
                    t_value=t_value,
                )
                arrow_latents, data_score = sample_critic_snapshot(
                    clean_latents=vector_clean_latents,
                    t_value=t_value,
                )
                score_figure = plot_critic_score_snapshot(
                    cloud_latents=cloud_latents,
                    cloud_labels=dense_labels,
                    arrow_latents=arrow_latents,
                    arrow_labels=vector_labels,
                    data_score=data_score,
                    t_value=t_value,
                )
                write_plot_artifacts(
                    figure=score_figure,
                    step=step,
                    plot_type=f"score_snapshot_t_{t_value:.2f}".replace(".", "p"),
                    stage="critic_pretrain",
                )

            spectrum_t_values = critic_spectrum_t_values()
            transport_field = estimate_clean_transport_field(
                clean_latents=vector_clean_latents,
                t_values=spectrum_t_values,
            )
            transport_grid_points, transport_grid_xs, transport_grid_ys = build_latent_grid(
                points=dense_clean_latents,
                resolution=31,
            )
            transport_grid_field = estimate_clean_transport_field(
                clean_latents=transport_grid_points,
                t_values=spectrum_t_values,
            )
            transport_grid_projection = (
                transport_grid_points * transport_grid_field
            ).sum(dim=-1).reshape(
                transport_grid_ys.shape[0],
                transport_grid_xs.shape[0],
            )
            transport_figure = plot_transport_field(
                cloud_latents=dense_clean_latents,
                cloud_labels=dense_labels,
                arrow_latents=vector_clean_latents,
                arrow_labels=vector_labels,
                transport_field=transport_field,
                grid_xs=transport_grid_xs,
                grid_ys=transport_grid_ys,
                grid_transport_projection=transport_grid_projection,
            )
            write_plot_artifacts(
                figure=transport_figure,
                step=step,
                plot_type="transport",
                stage="critic_pretrain",
            )

            loss_spectrum = estimate_critic_loss_spectrum(
                clean_latents=dense_clean_latents,
                t_values=spectrum_t_values,
            )
            loss_spectrum_figure = plot_critic_loss_spectrum(
                loss_spectrum=loss_spectrum,
            )
            write_plot_artifacts(
                figure=loss_spectrum_figure,
                step=step,
                plot_type="loss_spectrum",
                stage="critic_pretrain",
            )

    if encoder_was_training:
        encoder.train()
    if critic_was_training:
        critic.train()


## Pretrain

In [6]:
pretrain_history = []
encoder.train()
decoder.train()
critic.eval()

pretrain_progress = tqdm(
    range(1, chart_transport_config.scheduling_config.pretrain_chart_n_steps + 1),
    desc="pretrain",
)
for step in pretrain_progress:
    optimizer.zero_grad(set_to_none=True)

    with contextlib.ExitStack() as stack:
        stack.enter_context(torch.cuda.device(device))
        stack.enter_context(torch.device(str(device)))
        stack.enter_context(torch.autocast(device_type=device_type, dtype=torch.bfloat16))

        data_batch = runtime_data_config.sample_unconditional(
            batch_size=training_config.train_batch_size,
        )
        prior_batch = prior_config.sample(
            batch_size=training_config.train_batch_size,
        ).to(device=device, dtype=torch.float32)

        data_latents = encoder(data_batch)
        data_reconstruction = decoder(data_latents)
        prior_reconstruction = decoder(prior_batch)
        prior_latents = encoder(prior_reconstruction)

        data_cycle_loss = F.huber_loss(
            data_reconstruction,
            data_batch,
            delta=constraint_config.huber_delta,
            reduction="mean",
        )
        prior_cycle_loss = F.huber_loss(
            prior_latents,
            prior_batch,
            delta=constraint_config.huber_delta,
            reduction="mean",
        )
        constraint_loss = data_cycle_loss + prior_cycle_loss

        zero_mean_loss = F.huber_loss(
            data_latents.mean(),
            torch.zeros((), device=device, dtype=data_latents.dtype),
            delta=1.0,
            reduction="mean",
        )
        latent_norms = data_latents.square().sum(dim=-1).sqrt()
        softplus_loss = F.softplus(
            latent_norms - pretrain_config.softplus_radius,
        ).mean()
        chart_loss = constraint_loss
        chart_loss = chart_loss + pretrain_config.zero_mean_weight * zero_mean_loss
        chart_loss = chart_loss + pretrain_config.softplus_weight * softplus_loss

    fabric.backward(chart_loss)
    fabric.clip_gradients(
        chart_transport_model,
        optimizer,
        max_norm=architecture_config.grad_clip_norm,
    )
    optimizer.step()

    metrics = {
        "stage": "pretrain",
        "step": step,
        "chart_loss": chart_loss.detach().item(),
        "data_cycle_loss": data_cycle_loss.detach().item(),
        "prior_cycle_loss": prior_cycle_loss.detach().item(),
        "zero_mean_loss": zero_mean_loss.detach().item(),
        "softplus_loss": softplus_loss.detach().item(),
    }
    pretrain_history.append(metrics)
    pretrain_progress.set_postfix(
        chart_loss=f"{metrics['chart_loss']:.4f}",
        data_cycle=f"{metrics['data_cycle_loss']:.4f}",
        prior_cycle=f"{metrics['prior_cycle_loss']:.4f}",
    )

    if (
        step == 1
        or step % monitor_config.pretrain_monitor_config.log_every_n_steps == 0
        or step == chart_transport_config.scheduling_config.pretrain_chart_n_steps
    ):
        summary = ", ".join(
            f"{key}={value:.4f}"
            for key, value in metrics.items()
            if isinstance(value, float)
        )
        fabric.print(f"[pretrain] step {step}/{chart_transport_config.scheduling_config.pretrain_chart_n_steps}: {summary}")
        monitor_pretrain_step(step=step)

pretrain_history = pl.DataFrame(pretrain_history)
display(pretrain_history.tail())

pretrain:   0%|          | 0/1000 [00:00<?, ?it/s]

[pretrain] step 1/1000: chart_loss=0.6847, data_cycle_loss=0.1567, prior_cycle_loss=0.5276, zero_mean_loss=0.0202, softplus_loss=0.0152
[pretrain] step 100/1000: chart_loss=0.0010, data_cycle_loss=0.0002, prior_cycle_loss=0.0004, zero_mean_loss=0.0012, softplus_loss=0.0381
[pretrain] step 200/1000: chart_loss=0.0017, data_cycle_loss=0.0004, prior_cycle_loss=0.0010, zero_mean_loss=0.0001, softplus_loss=0.0324
[pretrain] step 300/1000: chart_loss=0.0010, data_cycle_loss=0.0003, prior_cycle_loss=0.0004, zero_mean_loss=0.0000, softplus_loss=0.0291
[pretrain] step 400/1000: chart_loss=0.0006, data_cycle_loss=0.0001, prior_cycle_loss=0.0002, zero_mean_loss=0.0000, softplus_loss=0.0266
[pretrain] step 500/1000: chart_loss=0.0006, data_cycle_loss=0.0001, prior_cycle_loss=0.0002, zero_mean_loss=0.0000, softplus_loss=0.0251
[pretrain] step 600/1000: chart_loss=0.0004, data_cycle_loss=0.0000, prior_cycle_loss=0.0001, zero_mean_loss=0.0000, softplus_loss=0.0245
[pretrain] step 700/1000: chart_loss

stage,step,chart_loss,data_cycle_loss,prior_cycle_loss,zero_mean_loss,softplus_loss
str,i64,f64,f64,f64,f64,f64
"""pretrain""",996,0.000372,0.000066,0.000081,0.000078,0.022413
"""pretrain""",997,0.000444,0.000103,0.000118,0.000005,0.022299
"""pretrain""",998,0.000474,0.000128,0.00012,0.000032,0.022489
"""pretrain""",999,0.000501,0.00013,0.000146,0.000071,0.0224
"""pretrain""",1000,0.000472,0.000113,0.000132,3.8825e-7,0.022739


## Train noise critic

In [ ]:
critic_pretrain_history = []
encoder.eval()
decoder.eval()
critic.train()

critic_pretrain_progress = tqdm(
    range(1, chart_transport_config.scheduling_config.pretrain_critic_n_steps + 1),
    desc="critic_pretrain",
)
for step in critic_pretrain_progress:
    optimizer.zero_grad(set_to_none=True)

    with contextlib.ExitStack() as stack:
        if device_type == "cuda":
            stack.enter_context(torch.cuda.device(device))
        stack.enter_context(torch.device(str(device)))
        stack.enter_context(torch.autocast(device_type=device_type, dtype=torch.bfloat16))

        data_batch = runtime_data_config.sample_unconditional(
            batch_size=training_config.train_batch_size,
        )
        with torch.no_grad():
            data_latents = encoder(data_batch)

        t = sample_transport_times(
            batch_shape=(data_latents.shape[0],),
        )
        eps = torch.randn_like(data_latents)
        noised_latents = (
            (1.0 - t).unsqueeze(-1) * data_latents + t.unsqueeze(-1) * eps
        )
        predicted_noise = critic(noised_latents, t)
        critic_loss = F.mse_loss(predicted_noise, eps)

    fabric.backward(critic_loss)
    fabric.clip_gradients(
        chart_transport_model,
        optimizer,
        max_norm=architecture_config.grad_clip_norm,
    )
    optimizer.step()

    metrics = {
        "stage": "critic_pretrain",
        "step": step,
        "critic_loss": critic_loss.detach().item(),
    }
    critic_pretrain_history.append(metrics)
    critic_pretrain_progress.set_postfix(
        critic_loss=f"{metrics['critic_loss']:.4f}",
    )

    if (
        step == 1
        or step % monitor_config.critic_monitor_config.log_every_n_steps == 0
        or step == chart_transport_config.scheduling_config.pretrain_critic_n_steps
    ):
        summary = ", ".join(
            f"{key}={value:.4f}"
            for key, value in metrics.items()
            if isinstance(value, float)
        )
        fabric.print(
            f"[critic_pretrain] step {step}/{chart_transport_config.scheduling_config.pretrain_critic_n_steps}: {summary}"
        )
        monitor_critic_step(step=step)

critic_pretrain_history = pl.DataFrame(critic_pretrain_history)
display(critic_pretrain_history.tail())

critic_pretrain:   0%|          | 0/1000 [00:00<?, ?it/s]

[critic_pretrain] step 1/1000: critic_loss=0.8986
[critic_pretrain] step 100/1000: critic_loss=0.2578


## Integrated training

In [ ]:
integrated_history = []
critic_steps_per_chart_step = max(
    1,
    chart_transport_config.scheduling_config.update_chart_every_n_critic_steps,
)

integrated_progress = tqdm(
    range(1, run_config.integrated_n_steps + 1),
    desc="integrated",
)
for step in integrated_progress:
    encoder.eval()
    decoder.eval()
    critic.train()
    optimizer.zero_grad(set_to_none=True)

    with contextlib.ExitStack() as stack:
        if device_type == "cuda":
            stack.enter_context(torch.cuda.device(device))
        stack.enter_context(torch.device(str(device)))
        stack.enter_context(torch.autocast(device_type=device_type, dtype=torch.bfloat16))

        critic_data_batch = runtime_data_config.sample_unconditional(
            batch_size=training_config.train_batch_size,
        )
        with torch.no_grad():
            critic_data_latents = encoder(critic_data_batch)

        critic_t = sample_transport_times(
            batch_shape=(critic_data_latents.shape[0],),
        )
        critic_eps = torch.randn_like(critic_data_latents)
        critic_noised_latents = (
            (1.0 - critic_t).unsqueeze(-1) * critic_data_latents
            + critic_t.unsqueeze(-1) * critic_eps
        )
        critic_predicted_noise = critic(critic_noised_latents, critic_t)
        critic_loss = F.mse_loss(critic_predicted_noise, critic_eps)

    fabric.backward(critic_loss)
    fabric.clip_gradients(
        chart_transport_model,
        optimizer,
        max_norm=architecture_config.grad_clip_norm,
    )
    optimizer.step()

    metrics = {
        "stage": "integrated",
        "step": step,
        "critic_loss": critic_loss.detach().item(),
        "chart_loss": float("nan"),
        "encoder_transport_loss": float("nan"),
        "decoder_transport_loss": float("nan"),
        "transport_field_norm": float("nan"),
        "data_cycle_loss": float("nan"),
        "prior_cycle_loss": float("nan"),
        "data_dual": data_dual.detach().item(),
        "prior_dual": prior_dual.detach().item(),
    }

    if step % critic_steps_per_chart_step == 0:
        encoder.train()
        decoder.train()
        optimizer.zero_grad(set_to_none=True)

        with contextlib.ExitStack() as stack:
            if device_type == "cuda":
                stack.enter_context(torch.cuda.device(device))
            stack.enter_context(torch.device(str(device)))
            stack.enter_context(torch.autocast(device_type=device_type, dtype=torch.bfloat16))

            chart_data_batch = runtime_data_config.sample_unconditional(
                batch_size=training_config.train_batch_size,
            )
            chart_prior_batch = prior_config.sample(
                batch_size=training_config.train_batch_size,
            ).to(device=device, dtype=torch.float32)

            chart_data_latents = encoder(chart_data_batch)
            chart_data_reconstruction = decoder(chart_data_latents)
            chart_prior_reconstruction = decoder(chart_prior_batch)
            chart_prior_latents = encoder(chart_prior_reconstruction)

            data_cycle_loss = F.huber_loss(
                chart_data_reconstruction,
                chart_data_batch,
                delta=constraint_config.huber_delta,
                reduction="mean",
            )
            prior_cycle_loss = F.huber_loss(
                chart_prior_latents,
                chart_prior_batch,
                delta=constraint_config.huber_delta,
                reduction="mean",
            )

            if isinstance(constraint_method, LossConstraintConfig):
                constraint_loss = (
                    constraint_method.data_loss_weight * data_cycle_loss
                    + constraint_method.prior_loss_weight * prior_cycle_loss
                )
            else:
                constraint_loss = data_cycle_loss + prior_cycle_loss
                constraint_loss = constraint_loss + data_dual * (
                    data_cycle_loss - constraint_method.data_constraint_budget
                )
                constraint_loss = constraint_loss + prior_dual * (
                    prior_cycle_loss - constraint_method.prior_constraint_budget
                )

            with torch.no_grad():
                transport_source_latents = encoder(chart_data_batch)
                transport_t = sample_stratified_transport_times(
                    batch_size=transport_source_latents.shape[0],
                    num_time_samples=transport_config.num_time_samples,
                )

                transport_source_latents = transport_source_latents.unsqueeze(1).expand(
                    -1,
                    transport_config.num_time_samples,
                    -1,
                )
                transport_eps = torch.randn(
                    transport_source_latents.shape[0],
                    transport_config.num_time_samples,
                    transport_source_latents.shape[-1],
                    device=device,
                    dtype=transport_source_latents.dtype,
                )

                if transport_config.antipodal_estimate:
                    transport_t = torch.cat([transport_t, transport_t], dim=1)
                    transport_source_latents = transport_source_latents.repeat(1, 2, 1)
                    transport_eps = torch.cat([transport_eps, -transport_eps], dim=1)

                transport_noised_latents = (
                    (1.0 - transport_t).unsqueeze(-1) * transport_source_latents
                    + transport_t.unsqueeze(-1) * transport_eps
                )
                flat_transport_noised_latents = transport_noised_latents.reshape(
                    -1,
                    transport_noised_latents.shape[-1],
                )
                flat_transport_t = transport_t.reshape(-1)

                transport_predicted_noise = critic(
                    flat_transport_noised_latents,
                    flat_transport_t,
                ).reshape_as(transport_noised_latents)
                transport_prior_score = prior_config.analytic_score(
                    flat_transport_noised_latents.float(),
                    flat_transport_t.float(),
                ).to(dtype=flat_transport_noised_latents.dtype).reshape_as(
                    transport_noised_latents
                )
                transport_pullback_weight = (
                    transport_config.kl_weight_schedule.pullback_weight(
                        flat_transport_t.float(),
                    )
                    .to(dtype=flat_transport_noised_latents.dtype)
                    .reshape_as(transport_t)
                )
                transport_field_terms = transport_pullback_weight.unsqueeze(-1) * (
                    transport_prior_score
                    + transport_predicted_noise / transport_t.unsqueeze(-1)
                )

                if transport_config.antipodal_estimate:
                    midpoint = transport_config.num_time_samples
                    transport_field_terms = 0.5 * (
                        transport_field_terms[:, :midpoint]
                        + transport_field_terms[:, midpoint:]
                    )

                transport_field = transport_field_terms.mean(dim=1)
                transported_latents = (
                    encoder(chart_data_batch)
                    + transport_config.transport_step_size * transport_field
                )

            live_latents = encoder(chart_data_batch)
            encoder_transport_loss = F.huber_loss(
                live_latents,
                transported_latents,
                delta=transport_config.huber_delta,
                reduction="mean",
            )
            decoder_transport_loss = F.huber_loss(
                decoder(transported_latents),
                chart_data_batch,
                delta=transport_config.huber_delta,
                reduction="mean",
            )
            chart_loss = constraint_loss
            chart_loss = chart_loss + (
                transport_config.encoder_transport_weight * encoder_transport_loss
            )
            chart_loss = chart_loss + (
                transport_config.decoder_transport_weight * decoder_transport_loss
            )

        fabric.backward(chart_loss)
        fabric.clip_gradients(
            chart_transport_model,
            optimizer,
            max_norm=architecture_config.grad_clip_norm,
        )
        optimizer.step()

        if isinstance(constraint_method, LagrangianConstraintConfig):
            data_dual = (
                data_dual
                + constraint_method.dual_variable_lr
                * (data_cycle_loss.detach() - constraint_method.data_constraint_budget)
            ).clamp_min(0.0)
            prior_dual = (
                prior_dual
                + constraint_method.dual_variable_lr
                * (prior_cycle_loss.detach() - constraint_method.prior_constraint_budget)
            ).clamp_min(0.0)

        metrics["chart_loss"] = chart_loss.detach().item()
        metrics["encoder_transport_loss"] = encoder_transport_loss.detach().item()
        metrics["decoder_transport_loss"] = decoder_transport_loss.detach().item()
        metrics["transport_field_norm"] = transport_field.norm(dim=-1).mean().detach().item()
        metrics["data_cycle_loss"] = data_cycle_loss.detach().item()
        metrics["prior_cycle_loss"] = prior_cycle_loss.detach().item()
        metrics["data_dual"] = data_dual.detach().item()
        metrics["prior_dual"] = prior_dual.detach().item()

    integrated_history.append(metrics)
    integrated_progress.set_postfix(
        critic_loss=f"{metrics['critic_loss']:.4f}",
        chart_loss=(
            f"{metrics['chart_loss']:.4f}"
            if metrics['chart_loss'] == metrics['chart_loss']
            else "nan"
        ),
    )

    if (
        step == 1
        or step % monitor_config.integrated_monitor_config.log_every_n_steps == 0
        or step == run_config.integrated_n_steps
    ):
        summary = ", ".join(
            f"{key}={value:.4f}"
            for key, value in metrics.items()
            if isinstance(value, float)
        )
        fabric.print(f"[integrated] step {step}/{run_config.integrated_n_steps}: {summary}")

integrated_history = pl.DataFrame(integrated_history)
display(integrated_history.tail())


[integrated] step 1/4000: critic_loss=0.1576, chart_loss=0.0833, encoder_transport_loss=0.0333, decoder_transport_loss=0.0499, transport_field_norm=2.3915, data_cycle_loss=0.0000, prior_cycle_loss=0.0000, data_dual=0.0000, prior_dual=0.0000
[integrated] step 100/4000: critic_loss=0.5247, chart_loss=0.2406, encoder_transport_loss=0.1850, decoder_transport_loss=0.0365, transport_field_norm=4.0488, data_cycle_loss=0.0080, prior_cycle_loss=0.0111, data_dual=0.0016, prior_dual=0.0031
[integrated] step 200/4000: critic_loss=0.7094, chart_loss=0.3690, encoder_transport_loss=0.2906, decoder_transport_loss=0.0398, transport_field_norm=7.7314, data_cycle_loss=0.0150, prior_cycle_loss=0.0235, data_dual=0.0040, prior_dual=0.0092
[integrated] step 300/4000: critic_loss=0.5972, chart_loss=0.1981, encoder_transport_loss=0.1724, decoder_transport_loss=0.0166, transport_field_norm=4.8767, data_cycle_loss=0.0060, prior_cycle_loss=0.0031, data_dual=0.0047, prior_dual=0.0105
[integrated] step 400/4000: cr

stage,step,critic_loss,chart_loss,encoder_transport_loss,decoder_transport_loss,transport_field_norm,data_cycle_loss,prior_cycle_loss,data_dual,prior_dual
str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""integrated""",3996,0.468332,0.116312,0.095019,0.010869,3.248471,0.002366,0.008234,0.005442,0.076104
"""integrated""",3997,0.490563,0.145896,0.119451,0.01434,3.815339,0.002535,0.009638,0.005434,0.076104
"""integrated""",3998,0.473801,0.141739,0.109509,0.015998,4.116457,0.00337,0.012693,0.005428,0.076106
"""integrated""",3999,0.493203,0.167052,0.131264,0.018947,4.685313,0.004261,0.012425,0.005422,0.076109
"""integrated""",4000,0.474768,0.16794,0.131163,0.018817,4.76459,0.00442,0.013317,0.005417,0.076112
